# Notebook 3 — Network Topology Evolution and TERGM Analysis

This notebook has two distinct parts.

**Part A — Network Topology Evolution (Sections 3.1–3.3)**
Documents how the intra-EA production network structure evolved from 1995 to 2022,
with attention to the three structural breaks: 2009 GFC, 2020 COVID, and 2022
energy/geopolitical shock.

**Part B — Temporal ERGM (Sections 3.4–3.6)**
Tests whether ECB monetary policy shocks predict changes in the EA production
network structure, controlling for endogenous network dynamics (reciprocity,
transitivity, memory). This addresses the reverse causality question: does
monetary policy *change* the network, not just get moderated by it?

We use a **country-level aggregated network** (19 nodes) for tractability.
The full sector-level network (817 nodes) is computationally prohibitive for TERGM;
the country-level aggregation preserves the bilateral linkage structure while
enabling estimation in minutes rather than hours.

**Requirements:**
- Run Notebook 1 first.
- For TERGM: R with `btergm` package installed (`install.packages("btergm")`).
  A Python fallback using network change metrics runs if R is unavailable.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

from network_metrics import (
    load_metrics_panel, compute_country_level_topology,
    compute_eigenvector_centrality, compute_network_change,
    compute_structural_break_panel, detect_production_clusters,
)
from tiva_loader import EA_COUNTRIES_TIVA

DATA_PROC = Path('../data/processed')
RESULTS   = Path('../results/figures')
TABLES    = Path('../results/tables')
for p in [RESULTS, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

# Check for R availability (for TERGM)
import subprocess
try:
    result = subprocess.run(['Rscript', '--version'],
                            capture_output=True, text=True, timeout=10)
    HAS_R = result.returncode == 0
except Exception:
    HAS_R = False

print(f'R available: {HAS_R}')
if not HAS_R:
    print('TERGM will use Python fallback (network change regression).')
    print('To enable full TERGM: install R and run: install.packages("btergm")')

---
## 3.1 Load Network Panel

In [ ]:
# Load metrics panel built in Notebook 1
metrics_path = DATA_PROC / 'network_metrics_panel.parquet'
topo_path    = DATA_PROC / 'country_topology.parquet'

if not metrics_path.exists():
    raise FileNotFoundError('Run Notebook 1 first.')

metrics = load_metrics_panel(DATA_PROC)
topo    = pd.read_parquet(topo_path)

print(f'Metrics panel: {len(metrics):,} obs')
print(f'Years: {metrics.reset_index()["year"].min()} – '
      f'{metrics.reset_index()["year"].max()}')
print(f'Countries: {metrics.reset_index()["country"].nunique()}')
print(f'Sectors per country: '
      f'{metrics.reset_index().groupby("country")["sector"].nunique().mean():.1f}')

---
## 3.2 Network Structure Over Time

Three key trends: declining maximum centrality (supply chain diversification), structural breaks at 2009/2020/2022, and persistence of Germany as the most central economy.

In [ ]:
ct = topo.reset_index()
EXCLUDE = {'CYP', 'MLT'}  # too small for meaningful topology trends
ct = ct[~ct['country'].isin(EXCLUDE)]

COLORS = {
    'DEU':'#FF6B35', 'FRA':'#4CC9F0', 'ITA':'#7B2FBE',
    'ESP':'#FFBE0B', 'NLD':'#FF006E', 'IRL':'#34D399',
    'BEL':'#06D6A0', 'AUT':'#FB5607',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

BREAKS = [2009, 2020, 2022]
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9')
    for sp in ax.spines.values(): sp.set_color('#30363d')
    for yr in BREAKS:
        ax.axvline(yr, color='#FF006E', ls='--', lw=1.2, alpha=0.5,
                   label='Structural break' if yr == BREAKS[0] else '')

# Panel 1: NCI over time
for country, color in COLORS.items():
    sub = ct[ct['country'] == country].sort_values('year')
    if len(sub) > 0:
        axes[0].plot(sub['year'], sub['nci'], color=color, lw=2,
                     label=country, marker='o', ms=2)
axes[0].set_title('Network Centralisation Index (Gini)', color='white', fontsize=11)
axes[0].set_ylabel('NCI', color='#c9d1d9')
axes[0].legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=7, ncol=2)

# Panel 2: Max forward linkage over time (EA-wide)
# = share of total intra-EA intermediate exports held by most central sector
max_fl = (metrics.reset_index()
          .groupby('year')['eigenvector_cent']
          .max()
          .reset_index())
axes[1].plot(max_fl['year'], max_fl['eigenvector_cent'],
             color='#FF6B35', lw=2.5, marker='o', ms=4)
axes[1].fill_between(max_fl['year'], max_fl['eigenvector_cent'],
                      alpha=0.15, color='#FF6B35')
axes[1].set_title('Maximum Sector Forward Linkage (EA-wide)
'
                   'Declining = production network becoming more distributed',
                   color='white', fontsize=10)
axes[1].set_ylabel('Top sector share of total intra-EA flows', color='#c9d1d9')

# Panel 3: Mean upstreamness
for country, color in COLORS.items():
    sub = ct[ct['country'] == country].sort_values('year')
    if len(sub) > 0:
        axes[2].plot(sub['year'], sub['mean_upstreamness'],
                     color=color, lw=2, label=country, marker='o', ms=2)
axes[2].set_title('Mean Upstreamness by Country', color='white', fontsize=11)
axes[2].set_ylabel('Upstreamness (1=downstream, 5=upstream)', color='#c9d1d9')
axes[2].legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=7, ncol=2)

# Panel 4: Edge count over time (load from saved data or approximate)
metrics_r = metrics.reset_index()
yearly_nodes = metrics_r.groupby('year')['sector'].count()
axes[3].plot(yearly_nodes.index, yearly_nodes.values,
             color='#8338EC', lw=2.5, marker='o', ms=4)
axes[3].set_title('Network Size (country-sector nodes × year)',
                   color='white', fontsize=11)
axes[3].set_ylabel('Total node-years', color='#c9d1d9')

for ax in axes:
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.setp(ax.xaxis.get_majorticklabels(), color='#c9d1d9')

plt.tight_layout()
plt.savefig(RESULTS / 'network_evolution.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 3.3 Community Structure

Which sectors cluster together, and how stable are these clusters across structural break years?

In [ ]:
# Load community assignments from metrics panel if available
# Otherwise compute for three snapshot years
m_cols = metrics.columns.tolist()
has_community = 'community_id' in m_cols

if has_community:
    comm = metrics.reset_index()
    for yr in [2000, 2010, 2022]:
        yr_comm = comm[comm['year'] == yr]
        if len(yr_comm) == 0:
            continue
        n_comm = yr_comm['community_id'].nunique()
        sizes  = yr_comm.groupby('community_id').size().sort_values(ascending=False)
        top3   = sizes.head(3).values
        print(f'{yr}: {n_comm} communities | '
              f'top 3 sizes: {top3}')

        # Which countries dominate each top community?
        for comm_id in sizes.head(3).index:
            members = yr_comm[yr_comm['community_id'] == comm_id]
            countries = members['country'].value_counts().head(4).index.tolist()
            sectors   = members['sector'].value_counts().head(3).index.tolist()
            print(f'  Community {comm_id}: countries={countries} | '
                  f'sectors={sectors}')
        print()
else:
    print('Community assignments not in metrics panel.')
    print('Set compute_communities=True in Notebook 1 and regenerate.')

---
## 3.4 Country-Level Aggregated Network (TERGM Input)

We aggregate the sector-level network to 19 country nodes, where edge weight
= total intermediate exports from country A to country B across all sectors.

**Why aggregate?**
TERGM on 817 nodes × 28 years requires MCMC sampling that takes hours.
At 19 nodes, each year's network is a 19×19 matrix — TERGM fits in minutes.
The aggregation is not a loss of information for the TERGM question
(does monetary policy predict bilateral country-level supply chain links?)
because the BLS and policy rate data are at the country level anyway.

**The binarisation decision:**
TERGM in its standard form (btergm) works on binary networks.
We define an edge from A→B as present if total intermediate exports exceed
the 50th percentile of all bilateral flows in that year. This captures whether
the supply chain relationship is *significant* (above median), not just present.

In [ ]:
# Build country-level aggregated network from ACTUAL bilateral flows
# Load the EXGR_INT parquet which has columns: country, partner, industry, year, value
# Aggregate to bilateral country pairs: total intermediate exports A→B per year

exgr_path = DATA_PROC / 'tiva_exgr_int.parquet'
if not exgr_path.exists():
    raise FileNotFoundError(
        'tiva_exgr_int.parquet not found in data/processed/. '
        'Run Notebook 1 first to generate it.'
    )

exgr = pd.read_parquet(exgr_path)
print(f'EXGR_INT loaded: {len(exgr):,} rows')
print(f'Columns: {list(exgr.columns)}')
print(f'Partner values sample: {exgr["partner"].value_counts().head(5).to_dict() if "partner" in exgr.columns else "no partner column"}')

# Aggregate to bilateral country-pair totals
# Filter: EA reporter → EA partner only (same as network construction)
SKIP = {"W", "WLD", "_Z", "OECD", "G20", "EA", "EU", "WORLD", "ROW", ""}
ea_set = set(EA_COUNTRIES_TIVA)

bilateral = (
    exgr[
        exgr["country"].isin(ea_set) &
        exgr["partner"].isin(ea_set) &
        ~exgr["partner"].isin(SKIP) &
        (exgr["country"] != exgr["partner"])
    ]
    .groupby(["country", "partner", "year"])["value"]
    .sum()
    .reset_index()
    .rename(columns={"value": "flow_usd_mn"})
)

print(f'\nBilateral panel: {len(bilateral):,} country-pair-year obs')
print(f'Years: {int(bilateral["year"].min())} – {int(bilateral["year"].max())}')
print(f'Unique pairs: {bilateral[["country","partner"]].drop_duplicates().shape[0]}')

# Build annual country-level networks from bilateral flows
def build_country_net_bilateral(bilateral_df, year, threshold_quantile=0.5):
    """Build binary country network from bilateral flows above median."""
    yr = bilateral_df[bilateral_df["year"] == year].copy()
    if len(yr) == 0:
        return None

    # Binarise: edge A→B if flow > median
    threshold = yr["flow_usd_mn"].quantile(threshold_quantile)
    yr_sig = yr[yr["flow_usd_mn"] >= threshold]

    G = nx.DiGraph()
    all_countries = sorted(ea_set &
                           set(yr["country"]) &
                           set(yr["partner"]))
    G.add_nodes_from(all_countries)

    for row in yr_sig.itertuples(index=False):
        G.add_edge(row.country, row.partner,
                   weight=float(row.flow_usd_mn))
    return G

years_net = sorted(bilateral["year"].dropna().unique().astype(int))
country_nets = {}
for yr in years_net:
    G = build_country_net_bilateral(bilateral, yr)
    if G is not None and G.number_of_edges() > 0:
        country_nets[yr] = G

country_nets_bin = country_nets  # already binary by threshold

print(f'\nCountry network panel: {len(country_nets)} years')
if country_nets:
    G_sample = list(country_nets.values())[len(country_nets)//2]
    yr_sample = list(country_nets.keys())[len(country_nets)//2]
    print(f'Sample ({yr_sample}): {G_sample.number_of_nodes()} nodes, '
          f'{G_sample.number_of_edges()} edges, '
          f'density={nx.density(G_sample):.3f}')

---
## 3.5 TERGM Analysis

### What TERGM Tests

Temporal ERGM models the probability of observing network G_t conditional on G_{t-1}:

```
P(G_t | G_{t-1}) ∝ exp(Σ θ_k · s_k(G_t, G_{t-1}))
```

Where s_k are network statistics:
- **edges**: baseline edge propensity
- **mutual**: reciprocity — if A→B, does B→A? (do bilateral supply chains go both ways?)
- **gwesp**: geometrically weighted edge-shared partners — transitivity (do A→B and B→C imply A→C?)
- **stability**: edges that existed in G_{t-1} — network memory / persistence
- **ECB rate change**: does monetary tightening predict fewer edges? (supply chain contraction)
- **NCI node covariate**: do high-NCI countries form/break links differently?

**The novel test:** the interaction between rate changes and NCI in edge formation.
If θ(rate × NCI) < 0: high-NCI countries *contract* their supply chain links more
when rates rise — paradoxically, the insulated country is also the one that restructures.
If θ(rate × NCI) > 0: high-NCI countries maintain their links better under monetary stress.
The latter would confirm and extend the transmission finding: network insulation operates
both in credit markets (BLS result) and in actual supply chain relationships.

In [ ]:
if HAS_R:
    import tempfile, os

    # Prepare data for TERGM
    countries_sorted = sorted(EA_COUNTRIES_TIVA)
    n_countries = len(countries_sorted)
    country_idx = {c: i for i, c in enumerate(countries_sorted)}

    rate_df = pd.read_parquet(DATA_PROC / 'ecb_policy_rate.parquet')
    policy_rate_r = rate_df.iloc[:, 0]
    policy_rate_r.index = pd.to_datetime(policy_rate_r.index)
    annual_rate_r = policy_rate_r.resample('YS').mean()
    annual_rate_r.index = annual_rate_r.index.year

    topo_r = topo.reset_index()

    adj_dir = DATA_PROC / 'tergm_adj'
    adj_dir.mkdir(exist_ok=True)

    valid_years = [yr for yr in sorted(country_nets_bin.keys())]

    for yr in valid_years:
        G = country_nets_bin[yr]
        adj = np.zeros((n_countries, n_countries), dtype=int)
        for u, v in G.edges():
            if u in country_idx and v in country_idx:
                adj[country_idx[u], country_idx[v]] = 1
        pd.DataFrame(adj, index=countries_sorted,
                     columns=countries_sorted).to_csv(adj_dir / f'adj_{yr}.csv')

    nci_mat = pd.DataFrame(index=valid_years, columns=countries_sorted, dtype=float)
    for yr in valid_years:
        yr_topo = topo_r[topo_r['year'] == yr].set_index('country')
        for c in countries_sorted:
            nci_mat.loc[yr, c] = yr_topo['nci'].get(c, np.nan) if 'nci' in yr_topo.columns else np.nan
    nci_mat.fillna(nci_mat.mean()).to_csv(DATA_PROC / 'tergm_nci_covariate.csv')

    rate_changes = {}
    for yr in valid_years:
        r_curr = annual_rate_r.get(yr, np.nan)
        r_prev = annual_rate_r.get(yr-1, np.nan)
        rate_changes[yr] = float(r_curr - r_prev) if not (np.isnan(r_curr) or np.isnan(r_prev)) else 0.0
    pd.Series(rate_changes, name='rate_change').to_csv(
        DATA_PROC / 'tergm_rate_changes.csv', header=True)

    # Build correct R vector strings
    years_r     = ', '.join(str(y) for y in valid_years)
    countries_r = ', '.join(f'"{c}"' for c in countries_sorted)
    adj_dir_r   = adj_dir.as_posix()
    nci_csv     = (DATA_PROC / 'tergm_nci_covariate.csv').as_posix()
    rate_csv    = (DATA_PROC / 'tergm_rate_changes.csv').as_posix()
    results_txt = (TABLES / 'tergm_results.txt').as_posix()

    r_script = f"""
library(btergm)
library(network)

adj_dir   <- "{adj_dir_r}"
years     <- c({years_r})
countries <- c({countries_r})
n         <- length(countries)

nci_mat  <- as.matrix(read.csv("{nci_csv}", row.names=1))
rate_raw <- read.csv("{rate_csv}", row.names=1)
rate_chg <- setNames(rate_raw[,1], as.character(years))

nets <- lapply(years, function(yr) {{
  adj <- as.matrix(read.csv(file.path(adj_dir, paste0("adj_", yr, ".csv")), row.names=1))
  net <- as.network(adj, directed=TRUE, matrix.type="adjacency")
  nci_yr <- nci_mat[as.character(yr), ]
  nci_yr[is.na(nci_yr)] <- mean(nci_yr, na.rm=TRUE)
  set.vertex.attribute(net, "nci", as.numeric(nci_yr))
  net
}})
names(nets) <- as.character(years)

cat("Fitting TERGM (may take several minutes)...\n")
set.seed(42)
tryCatch({{
  # timecov requires a list of n×n matrices (one per time step)
  # For a scalar covariate (same value for all dyads), create
  # a constant matrix filled with the rate change value
  rate_mats <- lapply(seq_along(nets), function(t) {{
    m <- matrix(rate_chg[t], nrow=n, ncol=n)
    diag(m) <- 0
    m
  }})

  model <- btergm(
    nets ~ edges + mutual + gwesp(0, fixed=TRUE) +
           memory(1) + nodecov("nci") +
           edgecov(rate_mats),
    R=500, parallel="no"
  )
  cat("\n=== TERGM RESULTS ===\n")
  print(summary(model))
  sink("{results_txt}")
  print(summary(model))
  sink()
  cat("Saved to {results_txt}\n")
}}, error=function(e) {{
  cat("Full model failed:", conditionMessage(e), "\nTrying simpler spec...\n")
  rate_mats2 <- lapply(seq_along(nets), function(t) {{
    m <- matrix(rate_chg[t], nrow=n, ncol=n)
    diag(m) <- 0
    m
  }})
  model2 <- btergm(
    nets ~ edges + mutual + memory(1) +
           nodecov("nci") +
           edgecov(rate_mats2),
    R=200, parallel="no"
  )
  print(summary(model2))
}})
"""

    r_path = DATA_PROC / 'run_tergm.R'
    with open(r_path, 'w') as f:
        f.write(r_script)
    print(f'R script written: {r_path}')
    print('Running TERGM...')

    result = subprocess.run(['Rscript', str(r_path)],
                            capture_output=True, text=True, timeout=1200)
    print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1500:])
else:
    print('R not available — TERGM skipped. See Section 3.6 for Python fallback.')

---
## 3.6 Python Fallback: Network Change Regression

If R is unavailable, we test the same hypothesis using a panel regression on network change metrics. For each consecutive year pair, we compute the Jaccard similarity of the edge set (how much of the network persisted) and the rank correlation of centrality rankings. We then regress these on the ECB rate change and lagged NCI.

In [ ]:
# This runs regardless of R availability — it is a complementary analysis
# even when TERGM is available

# Compute year-over-year network changes
rate_df = pd.read_parquet(DATA_PROC / 'ecb_policy_rate.parquet')
policy_rate_s = rate_df.iloc[:, 0]
policy_rate_s.index = pd.to_datetime(policy_rate_s.index)
annual_rate_s = policy_rate_s.resample('YS').mean()
annual_rate_s.index = annual_rate_s.index.year

change_records = []
sorted_years = sorted(country_nets.keys())
for yr_prev, yr_curr in zip(sorted_years[:-1], sorted_years[1:]):
    G_prev = country_nets[yr_prev]
    G_curr = country_nets[yr_curr]

    # Edge Jaccard: fraction of previous edges that persist
    e_prev = set(G_prev.edges())
    e_curr = set(G_curr.edges())
    jaccard = len(e_prev & e_curr) / (len(e_prev | e_curr) + 1e-9)

    # Centrality rank correlation
    ec_prev = nx.pagerank(G_prev, weight='weight', alpha=0.85)
    ec_curr = nx.pagerank(G_curr, weight='weight', alpha=0.85)
    common  = list(set(ec_prev) & set(ec_curr))
    if len(common) >= 5:
        rank_corr, _ = spearmanr(
            [ec_prev[c] for c in common],
            [ec_curr[c] for c in common]
        )
    else:
        rank_corr = np.nan

    # Density change
    d_change = nx.density(G_curr) - nx.density(G_prev)

    # ECB rate change
    r_prev = annual_rate_s.get(yr_prev, np.nan)
    r_curr = annual_rate_s.get(yr_curr, np.nan)
    delta_r = r_curr - r_prev if not np.isnan(r_prev + r_curr) else 0.0

    # NCI of the previous year (predetermined)
    topo_prev = topo.reset_index()
    topo_prev = topo_prev[topo_prev['year'] == yr_prev]
    mean_nci  = topo_prev['nci'].mean() if len(topo_prev) > 0 else np.nan

    change_records.append({
        'year':          yr_curr,
        'jaccard':       jaccard,
        'rank_corr':     rank_corr,
        'd_change':      d_change,
        'delta_rate':    delta_r,
        'nci_lag1':      mean_nci,
        'rate_x_nci':    delta_r * mean_nci if not np.isnan(mean_nci) else np.nan,
        'is_tightening': delta_r > 0,
    })

changes_df = pd.DataFrame(change_records).dropna()
print(f'Network change panel: {len(changes_df)} year-pairs')
print(changes_df[['year','jaccard','rank_corr','delta_rate','nci_lag1']].round(4).to_string())

In [ ]:
# Regression: does tightening predict network disruption?
import statsmodels.formula.api as smf

print('\n=== NETWORK CHANGE REGRESSIONS ===')
print()
print('Outcome: Jaccard similarity (higher = more stable network)')
print('Hypothesis: tightening → lower Jaccard (more disruption)')
print('            tightening × high NCI → ?')
print()

for outcome in ['jaccard', 'rank_corr', 'd_change']:
    if outcome not in changes_df.columns:
        continue
    df_r = changes_df.dropna(subset=[outcome, 'delta_rate', 'nci_lag1'])
    if len(df_r) < 10:
        continue

    # Model 1: rate change only
    try:
        m1 = smf.ols(f'{outcome} ~ delta_rate', data=df_r).fit()
        b1 = m1.params.get('delta_rate', np.nan)
        p1 = m1.pvalues.get('delta_rate', np.nan)
        s1 = '***' if p1<0.01 else '**' if p1<0.05 else '*' if p1<0.10 else ''
        print(f'{outcome} ~ delta_rate:          '
              f'β={b1:.4f}{s1}  p={p1:.4f}  N={int(m1.nobs)}')
    except Exception as e:
        print(f'{outcome} ~ delta_rate: {e}')

    # Model 2: rate change + NCI interaction
    try:
        m2 = smf.ols(f'{outcome} ~ delta_rate + nci_lag1 + rate_x_nci',
                     data=df_r).fit()
        b2 = m2.params.get('rate_x_nci', np.nan)
        p2 = m2.pvalues.get('rate_x_nci', np.nan)
        s2 = '***' if p2<0.01 else '**' if p2<0.05 else '*' if p2<0.10 else ''
        print(f'{outcome} ~ rate × NCI:           '
              f'β_int={b2:.4f}{s2}  p={p2:.4f}  N={int(m2.nobs)}')
    except Exception as e:
        print(f'{outcome} ~ rate × NCI: {e}')
    print()

changes_df.to_csv(TABLES / 'network_change_panel.csv', index=False)

# Plot: jaccard over time with tightening episodes highlighted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
TIGHTENING_YEARS = [[2000,2001],[2006,2007],[2011],[2022,2023]]

for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9')
    for sp in ax.spines.values(): sp.set_color('#30363d')

tight_years_flat = [y for group in TIGHTENING_YEARS for y in group]

colors_scatter = ['#FF006E' if y in tight_years_flat else '#4CC9F0'
                  for y in changes_df['year']]
axes[0].scatter(changes_df['year'], changes_df['jaccard'],
                color=colors_scatter, s=60, alpha=0.85, zorder=3)
axes[0].plot(changes_df['year'], changes_df['jaccard'],
             color='#5a7080', lw=1, alpha=0.5)
axes[0].set_title('Network Edge Stability (Jaccard)
Red = tightening years',
                   color='white', fontsize=11)
axes[0].set_ylabel('Jaccard similarity (t-1 → t)', color='#c9d1d9')

axes[1].scatter(changes_df['delta_rate'], changes_df['jaccard'],
                c=changes_df['nci_lag1'], cmap='RdYlGn_r',
                s=70, alpha=0.85, edgecolors='#30363d', zorder=3)
from scipy.stats import linregress
sl, ic, r, p, _ = linregress(changes_df['delta_rate'],
                               changes_df['jaccard'])
xl = np.linspace(changes_df['delta_rate'].min(),
                  changes_df['delta_rate'].max(), 50)
axes[1].plot(xl, sl*xl+ic, color='#FF6B35', lw=2, ls='--',
             label=f'β={sl:.3f}  r²={r**2:.3f}  p={p:.3f}')
axes[1].set_title('Rate Change vs Network Stability
(colour = NCI level)',
                   color='white', fontsize=11)
axes[1].set_xlabel('ECB rate change (pp)', color='#c9d1d9')
axes[1].set_ylabel('Jaccard similarity', color='#c9d1d9')
axes[1].legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=9)
axes[1].axvline(0, color='white', ls='--', lw=0.8, alpha=0.4)

plt.tight_layout()
plt.savefig(RESULTS / 'network_stability.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 3.7 Interpretation

### What TERGM / Network Change Regressions Test

The main regression in Notebook 4 asks: **does network topology predict transmission?**
Direction of causality: network → credit market outcomes.

This notebook asks the reverse: **does monetary policy reshape the network?**
Direction of causality: policy rate → network structure.

**If both results hold**, the project tells a richer, bidirectional story:

```
Higher NCI → Weaker credit tightening (Notebook 4)
ECB rate hikes → Network disruption (Notebook 3)
Higher NCI → More/less network disruption under rate hikes (interaction, Notebook 3)
```

The two-way relationship is more interesting than either direction alone.
A country like Ireland (high NCI, pharma-dominated) may transmit policy
less through credit markets, but its network position may also make it
more vulnerable to supply chain restructuring when rates rise sharply —
because its few supply chain partners matter disproportionately.

### Reading the Jaccard Results

Jaccard < 0 in tightening years would mean: rate hikes predict supply chain
disruption (fewer edges persist year-over-year). The interaction with NCI
tells you whether concentrated production structures are more or less vulnerable.

This is the **novel contribution** relative to standard monetary transmission
papers: production networks are not just a moderator of financial transmission,
they are themselves restructured by monetary policy shocks.